# Page 5 — "How to Reason": what the reasoning is worth

Three prompts on one model, on the same problems, greedy throughout. Produces everything page 5
renders; the page replays committed logs and runs no model.

**The finding.** A 2026 instruct model reasons on a maths question unprompted. You cannot make it
reason, because it already does. You *can* stop it, and stopping it costs about 36 points.

Writes `out/page_05/`: `traces.jsonl` (every generation), `derived.json` (what the page reads),
`manifest.json` (provenance), `figure_*.png` (reference renders the page redraws in the browser).

Condition design, item band and the claims contract live in
[`docs/page_5_refactor.md`](../docs/page_5_refactor.md). Shared plumbing lives in
[`harness.py`](harness.py), which ships alongside this notebook.

## How to run this

| Mode | Models | Items | Generations | Figure 3 | Hardware | Time |
|---|---|---|---|---|---|---|
| **`REPLAY`** *(default)* | none loaded | 100 | 0 — reads the published run | on | **no GPU** | **~1 min** |
| `QUICK` | Qwen2.5-3B-Instruct | 50 | 150 | off | ~6 GB GPU | tens of minutes |
| `FULL` | 3B + 1.5B + 7B | 100 | 900 | on | 16 GB+ GPU | **several hours** |

**Start with `REPLAY`.** It draws every figure from the generations the page ships, without
loading a model. It adopts the published `RUN_ID`, asserts the traces cover the item set drawn
below, and then runs every gate, assertion and figure the generating run did — including the
independent verification, which recomputes the headline numbers from the trace file. What it
cannot tell you is whether *your* hardware reproduces ours; that is what the other two modes are
for.

The 3B is the primary in both generating modes, so `QUICK` is a preview of `FULL` rather than a
different experiment. `FULL` adds a 1.5B and a 7B for figure 3 only and warns before it starts;
the 7B needs about 15 GB at bf16, and `FIGURE_3 = False` produces every other figure without it.

```bash
python -m venv .venv && source .venv/bin/activate
pip install "torch>=2.7" "transformers>=4.56" accelerate sympy matplotlib requests
pip install reasoning-from-scratch    # or: git+https://github.com/rasbt/reasoning-from-scratch.git
jupyter lab
```

Run from this directory with `harness.py` beside the notebook. MATH-500 is fetched on first run
into `data/`. No API key and no account; in `REPLAY`, no checkpoint either — set `TRACES_URL` in
the knobs cell or drop a `traces.jsonl` into `out/page_05/`.

Everything the page draws is in `derived.json`, and everything there is derived from
`traces.jsonl` by code in this notebook.

Resumable: `traces.jsonl` is appended as it goes and keyed by `RUN_ID`, which names the models,
band, cap and item set, so changing any of them starts a fresh set rather than replaying
incomparable traces. Delete the file to start over.

**Greedy decoding diverges on a handful of items across hardware** — floating-point reduction
order changes with the kernel, and ch02's `get_device()` enables TF32 on CUDA. Expect the shape
and the ranking to reproduce, not every digit. The manifest records the device.

# Part 1 — Setup and the sweep

Configuration, the item set, the prompts, the model, the generations and the checks on them.
Everything here produces `traces.jsonl` and nothing here draws anything.

**Skip to part 2** if what you came for is how the page's figures and numbers were made. In
`REPLAY` this part loads no model and takes about a minute; you are not skipping a wait.

In [ ]:
# ---------------------------------------------------------------------------
# The knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "REPLAY"       # "REPLAY" (no GPU), "QUICK" (one 3B), "FULL" (three models)
SEED = 123                # selects items only; decoding is greedy and unseeded

# The 3B is the primary in both modes: bf16 fits a 16 GB laptop, it sits at 43% bare
# so there is headroom in both directions, and unlike the 7B it needs no quantisation,
# so the page's headline number carries no precision caveat.
PRIMARY = "Qwen/Qwen2.5-3B-Instruct"

FULL_MODELS = [PRIMARY, "Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-7B-Instruct"]

PRESETS = {
    # Every figure on this page, drawn from the published generations. No GPU, no
    # weights downloaded, about a minute. Default, because checking a claim should not
    # cost a reader an afternoon and a 16 GB card.
    "REPLAY": dict(models=FULL_MODELS, n_items=100, figure_3=True),
    # One 3B on your own hardware: figures 1 and 2 at half the items. ~6 GB, tens of minutes.
    "QUICK":  dict(models=[PRIMARY],   n_items=50,  figure_3=False),
    # The shipped grid, regenerated from scratch. See the warning below.
    "FULL":   dict(models=FULL_MODELS, n_items=100, figure_3=True),
}
CFG = PRESETS[RUN_MODE]
MODELS = CFG["models"]
FIGURE_3 = CFG["figure_3"]          # set False if the 7B will not fit; nothing else changes

EVAL_LEVELS = (4, 5)      # MATH-500 holds 262 problems here
GEN_CAP = 2048            # one cap for every condition: the variable is the PROMPT.
                          # Asserted per condition below (spec section 3).

# Where REPLAY gets its generations. A local out/page_05/traces.jsonl wins if present,
# so a run you did yourself replays without touching the network.
TRACES_URL = None          # set to the published traces.jsonl once it is hosted

if RUN_MODE == "FULL":
    print("!" * 76)
    print("WARNING  A full reproduction makes 900 completions across three checkpoints,")
    print("         wants a 16 GB+ GPU (the 7B needs 8-bit below that), and takes")
    print("         several hours. RUN_MODE = \"REPLAY\" draws every figure on this")
    print("         page from the published traces in about a minute.")
    print("!" * 76 + "\n")

print(f"mode={RUN_MODE}  primary={PRIMARY}")
print(f"models={len(MODELS)}  items={CFG['n_items']}  cap={GEN_CAP}  "
      f"eval L{'+L'.join(map(str, EVAL_LEVELS))}  figure_3={FIGURE_3}")

In [ ]:
import json
import random
import re
from datetime import datetime, timezone
from math import comb, sqrt
from pathlib import Path

import torch
from tqdm.auto import tqdm

# Shared plumbing, shipped beside this notebook. What is here and what is
# deliberately not is documented at the top of the file.
import harness

# The inherited grading harness: text-in, text-out, no model dependency.
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test, render_prompt, extract_final_candidate, grade_answer)

OUT_DIR = Path("out/page_05"); OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data"); DATA_DIR.mkdir(parents=True, exist_ok=True)
TRACES_PATH = OUT_DIR / "traces.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

mean, median, wilson = harness.mean, harness.median, harness.wilson
print("torch", torch.__version__)
print("harness", harness.fingerprint()["version"], harness.fingerprint()["sha256"][:12])

## The grading harness

Inherited unchanged from ch03. `extract_final_candidate` takes the last `\boxed{...}`, falling
back to the last number in the trace; `grade_answer` checks equivalence with LaTeX canonicalisation
and SymPy. **Grading is not ours to defend**, which is the point of using it unchanged.

The fallback matters on this page: the blurt condition rarely emits `\boxed{}` by construction, so
its answers are extracted by that fallback. The integrity section reports the boxed rate per
condition so the reader can see where it applies.

In [ ]:
CHECKS = [
    # (raw completion, ground truth, expected)
    (r"The answer is \boxed{9}.",                          "9",     True),
    (r"Step 1: 23-20=3. Step 2: 3+6=9. So \boxed{9}.",     "9",     True),   # a chain
    (r"27",                                                 "9",     False),  # a wrong blurt
    (r"9",                                                  "9",     True),   # a right blurt, unboxed
    (r"\boxed{\frac{1}{2}}",                               "0.5",   True),   # LaTeX vs decimal
    (r"\boxed{1,024}",                                     "1024",  True),   # thousands separator
    (r"\boxed{\left( 3, \frac{\pi}{2} \right)}",           r"\left( 3, \frac{\pi}{2} \right)", True),
    (r"I am not sure.",                                     "9",     False),  # no answer at all
]

rows = [(t, g, e, bool(grade_answer(extract_final_candidate(t), g))) for t, g, e in CHECKS]
bad = [r for r in rows if r[3] != r[2]]
if bad:
    for t, g, e, got in rows:
        flag = "ok  " if got == e else "FAIL"
        print(f"{flag} {extract_final_candidate(t)!r:<38} vs {g!r:<12} -> {got}")
    raise AssertionError(f"{len(bad)} grader check(s) behaved unexpectedly")

print(f"grader preflight: {len(rows)}/{len(rows)} as expected "
      f"(boxed, chain-then-boxed, wrong blurt, unboxed blurt, LaTeX, separators, tuples, "
      f"no answer)")

## The item set

One draw, hashed, so "the same items" is checkable later and every model sees identical problems.
Every comparison on this page is therefore **paired at the problem level**, which is what lets the
figures use an exact McNemar test rather than comparing two independent intervals.

Levels 4 and 5, which is where the primary model sits at 43% and has room to move in both
directions. There is no exemplar band and no held-out pool: the worked-example conditions left the
ladder (spec section 3), and with them the most failure-prone cell this notebook used to have.

In [ ]:
math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")

band = [r for r in math500 if int(r["level"]) in EVAL_LEVELS]
band.sort(key=lambda r: r["unique_id"])          # sort before shuffling, or the draw
random.Random(SEED).shuffle(band)                 # depends on dict ordering
items = band[: CFG["n_items"]]

assert len(items) == CFG["n_items"], \
    f"asked for {CFG['n_items']} items, levels {EVAL_LEVELS} hold only {len(band)}"
ITEM_SET_HASH = harness.sha256_of(r["unique_id"] for r in items)

from collections import Counter
spread = "+".join(f"L{n}x{c}" for n, c in sorted(Counter(int(r["level"]) for r in items).items()))
print(f"scored {len(items)} of {len(band)} available at levels "
      f"{'+'.join(map(str, EVAL_LEVELS))}   {spread}   sha256 {ITEM_SET_HASH[:12]}")

## The three prompts

Every condition carries ch03's `render_prompt` stem unchanged, which fixes the `\boxed{}` answer
format. Vary that too and a condition could score lower only because the grader could not find its
answer.

| Condition | What it adds | Whose mechanism |
|---|---|---|
| `forced_blurt` | "Give only the final answer. No explanation, no working, no other text." | the floor, and this page's intervention |
| `bare` | nothing | the true default, which already chains |
| `zero_shot_cot` | "Explain step by step." | Kojima et al. 2022 · ch04 §4.3 |

The suffixes are word for word `experiment_3`'s, so the two trace banks are directly comparable;
the integrity section checks that. Exemplar conditions are excluded — spec §3 has the numbers.

In [ ]:
CONDITIONS = ["forced_blurt", "bare", "zero_shot_cot"]
LABEL = {"forced_blurt": "forced blurt", "bare": "bare",
         "zero_shot_cot": '+ "Explain step by step."'}

BLURT_SUFFIX = "\n\nGive only the final answer. No explanation, no working, no other text."
COT_SUFFIX   = "\n\nExplain step by step."      # ch04 section 4.3, verbatim mechanism


def build_prompt(condition, question):
    """The whole experiment, in one readable function. Nothing else varies."""
    stem = render_prompt(question)              # ch03's template -- CONSTANT across conditions
    if condition == "bare":
        return stem
    if condition == "forced_blurt":
        return stem + BLURT_SUFFIX
    return stem + COT_SUFFIX


# The prompts ARE the experiment, so print the shared stem once and then only what each
# condition adds and where. The page highlights the same diff, split on the same stem.
STEM = render_prompt("<QUESTION>")
print("the stem, identical in all three conditions")
print("-" * 74)
print(STEM)
print("-" * 74)
for c in CONDITIONS:
    added = build_prompt(c, "<QUESTION>")[len(STEM):].strip()
    print(f"  {c:<15} {'appends ' + repr(added) if added else 'adds nothing'}")

## Loading the model

**The cell below is the first one that runs the model.** Everything above it is text, arithmetic
and assertions; from here on there are weights in memory.

Four timed stages rather than one silent call, so a long wait is always attributable to a named
thing. The fetch downloads the checkpoint on the first run only (about 6 GB for the 3B) and is the
longest single wait; every run after that reads it from the cache in under a second. The warm-up
measures this machine's tokens per second, which the sweep uses to project the run before it
starts.

## Replaying the published run

**`REPLAY` reads generations instead of making them.** Nothing below knows the difference: the
same `results` structure is built either way, so every figure, gate and assertion runs
identically.

The item set is drawn above from the seed and band the published run used, and the replay asserts
the traces cover exactly those problems. A file that does not match says so rather than drawing a
figure from the wrong items.


In [ ]:
results, model_meta, SWEPT = {}, {}, []

if RUN_MODE == "REPLAY":
    if TRACES_URL and not TRACES_PATH.exists():
        print(f"fetching published generations from {TRACES_URL}")
        TRACES_PATH.write_text(requests.get(TRACES_URL, timeout=120).text, encoding="utf-8")

    assert TRACES_PATH.exists(), (
        f"REPLAY needs published generations at {TRACES_PATH} and found none. Set TRACES_URL "
        f"in the knobs cell, or use RUN_MODE = 'QUICK' to generate your own.")

    raw = [json.loads(l) for l in TRACES_PATH.open(encoding="utf-8") if l.strip()]
    assert raw, f"{TRACES_PATH} is empty"

    # Adopt the published run's id rather than computing one: a replay IS that run.
    RUN_ID = Counter(r["run_id"] for r in raw).most_common(1)[0][0]
    raw = [r for r in raw if r["run_id"] == RUN_ID]

    want = {r["unique_id"] for r in items}
    missing = want - {r["problem_id"] for r in raw}
    assert not missing, (
        f"the published traces miss {len(missing)} of this notebook's {len(want)} problems. "
        f"That file was made with a different seed, band or item count, and must not be "
        f"plotted against these items.")

    for r in raw:
        if r["problem_id"] in want:
            results.setdefault(r["model_id"], {}).setdefault(r["condition"], []).append(r)
    for mid, by_cond in results.items():
        for cond in by_cond:                       # stable item order across conditions
            by_cond[cond] = sorted(by_cond[cond], key=lambda r: r["problem_id"])
        # A replay has no weights to interrogate, so size and precision are read from
        # the checkpoint's name. Both are properties of the name, not measurements, and
        # the manifest records that this run replayed rather than loaded.
        label = mid.split("/")[-1]
        billions = float(re.search(r"(\d+(?:\.\d+)?)B", label).group(1))
        model_meta[mid] = dict(id=mid, label=label, params=int(billions * 1e9),
                               dtype=None, quant="8bit" if "8bit" in label.lower() else "bf16",
                               tokenizer_sha256=None, revision=None, replayed=True)

    SWEPT = [m for m in MODELS if m in results] or sorted(results)
    n = sum(len(v) for by in results.values() for v in by.values())
    print(f"replaying {RUN_ID}")
    print(f"  {n:,} generations, {len(SWEPT)} model(s), {len(want)} items, nothing loaded")
    for mid in SWEPT:
        print(f"    {mid.split('/')[-1]:<24}" +
              "  ".join(f"{c}={len(results[mid].get(c, []))}" for c in CONDITIONS))
else:
    print(f"{RUN_MODE}: generations are made below")


In [ ]:
if RUN_MODE != "REPLAY":
    device = get_device()
    DTYPE = harness.pick_dtype(device)

    loaded = harness.load_model(PRIMARY, device=device, dtype=DTYPE)
    model, tokenizer = loaded["model"], loaded["tokenizer"]
    N_PARAMS, WARMUP_TOK_S = loaded["params"], loaded["tok_per_s"]

    print()
    harness.selftest(tokenizer)      # the invariants every trace below depends on
else:
    device = DTYPE = model = tokenizer = loaded = None
    N_PARAMS, WARMUP_TOK_S = None, None
    print("REPLAY: no model loaded, no weights downloaded")

## Proof that the floor is real

**The cell to read if you suspect we cheated.** This page's headline is what happens when the
model is forbidden to reason, so the whole argument rests on the model actually obeying that
instruction. That is a property of the checkpoint, not something the page is entitled to assume,
and it is not universal: an earlier Qwen3 checkpoint obeyed on 5 items of 24, and the math-tuned
checkpoints refuse it outright.

So it is measured here, on this run's own items, before the sweep. A model that keeps reasoning
through an explicit instruction not to gives this page no floor, and the notebook says so and
gates on it rather than discovering it in the figures.

In [ ]:
COMPLIES_UNDER = 40      # tokens. A levels 4-5 answer is often not a bare number.
FLOOR_N = min(12, len(items))


def complete(prompt_text, mdl, tok, max_new_tokens=None):
    """One greedy completion through the model's own chat template. `stop` has no
    default in the harness, so the rule that ends an answer stays visible here."""
    chat = tok.apply_chat_template([{"role": "user", "content": prompt_text}],
                                   tokenize=False, add_generation_prompt=True)
    return harness.complete(chat, mdl, tok, stop=["\nQuestion:"],
                            max_new_tokens=max_new_tokens or GEN_CAP)


if RUN_MODE == "REPLAY":
    # The published run answered this already; measure it over every blurt it made
    # rather than over a sample of twelve.
    floor_lens = [r["tokens_generated"] for r in results[PRIMARY]["forced_blurt"]]
    FLOOR_N = len(floor_lens)
else:
    floor_lens = [complete(build_prompt("forced_blurt", r["problem"]), model, tokenizer,
                           max_new_tokens=128)["tokens_generated"]
                  for r in tqdm(items[:FLOOR_N], desc="blurt compliance", unit="item")]

BLURT_COMPLIANCE = mean([n <= COMPLIES_UNDER for n in floor_lens])
print(f"\n{sum(n <= COMPLIES_UNDER for n in floor_lens)}/{FLOOR_N} answered within "
      f"{COMPLIES_UNDER} tokens   median {median(floor_lens):.0f}, longest {max(floor_lens)}")
if BLURT_COMPLIANCE < 0.8:
    print("\n!! This model does not stop reasoning when told to, so the page has no floor and")
    print("   the suppression contrast is not measuring suppression. derived.json gates on")
    print("   this; see spec section 3.")
else:
    print("the floor is real: told to answer only, this model answers only")

## The sweep

One function: hand it a model, a condition and the items, and it returns the records.
`harness.TraceStore` writes each answer the moment it arrives and keys them by
`(model, condition, problem)`, so a stopped run loses nothing and a finished pass costs nothing to
re-run.

The record is the schema shared with pages 6 to 9, with `condition` where page 6 puts `budget`.
Provenance is not repeated per line; it lives once in `manifest.json`.

In [ ]:
if RUN_MODE != "REPLAY":
    # The key names everything that changes what a generation IS. Leaving the cap out of it
    # once meant that raising it and re-running silently replayed traces made under the old
    # cap, and reported the old truncation rate.
    RUN_ID = (f"p5-{RUN_MODE.lower()}-L{''.join(map(str, EVAL_LEVELS))}"
              f"-cap{GEN_CAP}-n{len(items)}-{ITEM_SET_HASH[:8]}")
    STORE = harness.TraceStore(TRACES_PATH, RUN_ID, ("model_id", "condition", "problem_id"))
    STORE.report()

    _gens = len(MODELS) * len(CONDITIONS) * len(items)
    print(f"\nabout {_gens:,} generations; at ~{WARMUP_TOK_S:.0f} tok/s and a {GEN_CAP}-token cap "
          f"that is under {_gens * GEN_CAP / max(WARMUP_TOK_S, 1e-6) / 3600:.1f} h worst case,\n"
          f"and far less in practice because most generations stop well short of the cap.\n")

    # results / model_meta come from the replay cell above


    def sweep_model(model_id, mdl, tok, n_params):
        results[model_id] = {}
        model_meta[model_id] = dict(
            id=model_id, label=model_id.split("/")[-1], params=n_params,
            dtype=str(next(mdl.parameters()).dtype),
            quant="bf16" if "int8" not in str(next(mdl.parameters()).dtype) else "8bit",
            tokenizer_sha256=harness.tokenizer_fingerprint(tok),
            revision=harness.repo_revision(model_id))
        for cond in CONDITIONS:
            recs = []
            bar = tqdm(items, desc=f"{model_id.split('/')[-1][:20]:<20} {cond:<15}", unit="item")
            for row in bar:
                rec = STORE.get(model_id, cond, row["unique_id"])
                if rec is None:
                    prompt = build_prompt(cond, row["problem"])
                    g = complete(prompt, mdl, tok)
                    got = extract_final_candidate(g["text"])
                    rec = STORE.append({
                        "model_id": model_id, "condition": cond,
                        "problem_id": row["unique_id"], "level": int(row["level"]),
                        "subject": row["subject"], "question": row["problem"],
                        "ground_truth": row["answer"], "prompt": prompt,
                        "trace": g["text"], "tokens": g["tokens"],
                        "truncated": g["truncated"], "max_new_tokens": GEN_CAP,
                        "extracted_answer": got,
                        "correct": bool(grade_answer(got, row["answer"])),
                        "tokens_generated": g["tokens_generated"],
                        "boxed": "\\boxed" in g["text"],
                    })
                recs.append(rec)
                bar.set_postfix(acc=f"{mean([r['correct'] for r in recs]):.2f}")
            results[model_id][cond] = recs


    sweep_model(PRIMARY, model, tokenizer, N_PARAMS)

In [ ]:
# Figure 3's extra models, one resident at a time. The primary is freed first, so peak
# memory is one checkpoint rather than three. Everything above this cell is already
# computed, so a machine that cannot hold the 7B still produces figures 1 and 2.
if RUN_MODE != "REPLAY" and FIGURE_3 and len(MODELS) > 1:
    del model, loaded
    harness.free_memory()
    for mid in MODELS[1:]:
        try:
            got = harness.load_model(mid, device=device, dtype=DTYPE)
            sweep_model(mid, got["model"], got["tokenizer"], got["params"])
            del got
            harness.free_memory()
        except Exception as e:          # OOM is the expected one; any load failure is survivable
            print(f"\n!! {mid} did not load ({type(e).__name__}). Figure 3 will be drawn "
                  f"from the models that did.\n   {str(e)[:160]}")
            harness.free_memory()
    PRIMARY_LOADED = False
else:
    PRIMARY_LOADED = RUN_MODE != "REPLAY"
    if RUN_MODE != "REPLAY":
        print("one model only; figure 3 is off")

SWEPT = [m for m in MODELS if m in results] or sorted(results)
print("\nswept:", ", ".join(m.split("/")[-1] for m in SWEPT))

## Reasoning steps, and where they sit in the token stream

The rule, in order: blank-line paragraphs, then newlines, then sentences — the first that actually
divides the trace. Fragments under `MIN_STEP_CHARS` merge backwards. Steps carry contiguous token
ranges covering every token, so the page's reveal stops at real boundaries.

**The rule counts formatting, not reasoning** — a display equation is its own step. It places the
reveal boundaries and detects blurts. It is not a measure of how much reasoning happened and must
not be reported as one.

In [ ]:
MIN_STEP_CHARS = 12

_CACHE = {}
def segment_steps_cached(rec):
    k = (rec["model_id"], rec["condition"], rec["problem_id"])
    if k not in _CACHE:
        _CACHE[k] = harness.segment_steps(rec["trace"], rec["tokens"], min_step_chars=MIN_STEP_CHARS)
    return _CACHE[k]

# Checked against a real trace from the sweep, not a toy string.
_probe = next((r for r in results[PRIMARY]["bare"] if r["correct"]), results[PRIMARY]["bare"][0])
_steps = harness.segment_steps(_probe["trace"], _probe["tokens"], min_step_chars=MIN_STEP_CHARS)
_norm = lambda s: re.sub(r"\s+", "", s)
assert _norm("".join(s["text"] for s in _steps)) == _norm(_probe["trace"]), "segmentation dropped text"
assert _steps[-1]["token_end"] == len(_probe["tokens"]), "segmentation did not cover every token"
print(f"{_probe['problem_id']}: {_probe['tokens_generated']} tokens -> {len(_steps)} steps")
for s in _steps[:5]:
    print(f"  [{s['token_start']:>3}:{s['token_end']:>3}]  {s['text'][:84]}")

## Sweep integrity

Four checks the page's honesty rests on. The first two raise; the others report, because a low
boxed rate under the blurt is expected rather than a defect.

Truncation is checked **per condition**. An earlier run passed a 13% aggregate as a warning while
the cap clipped 0% of the answer-only condition and 33% of the step-by-step one, and a truncated
trace loses its `\boxed{}` and grades wrong, so that was a scoring penalty aimed at exactly the
conditions this page argues for.

In [ ]:
all_records = [r for m in SWEPT for c in CONDITIONS for r in results[m][c]]

# 1. no duplicates. TraceStore has no write lock, and two processes on one run_id
#    silently produce duplicate records and double-counted aggregates.
_keys = {(r["model_id"], r["condition"], r["problem_id"]) for r in all_records}
print(f"1. one record per (model, condition, problem) ... {len(all_records)} records, "
      f"{len(_keys)} keys")
assert len(all_records) == len(_keys), \
    "duplicate records: two processes wrote to this trace bank. Deduplicate before trusting it."

# 2. how hard the cap bound, PER CONDITION, and which way the bias runs.
#
#    The budget is 5% on the conditions the headline rests on and 10% on the reasoning
#    conditions, and the asymmetry is deliberate rather than a concession. A truncated
#    trace loses its \boxed{} and grades wrong, so truncation always costs the condition
#    it lands on. It lands on the reasoning conditions -- the blurt is six tokens and
#    cannot be cut -- so residual truncation *understates* bare and *shrinks* the
#    suppression effect. The measured worst case at GEN_CAP=2048 is 6% of Qwen2.5-3B's
#    bare items, six problems, biasing the page's headline conservatively.
#
#    A budget the other way round would be the dangerous one, and that is the version
#    this gate was originally written to catch.
TRUNCATION_BUDGET = {"forced_blurt": 0.05, "bare": 0.10, "zero_shot_cot": 0.10}
trunc = {(m, c): mean([r["truncated"] for r in results[m][c]]) for m in SWEPT for c in CONDITIONS}
worst = max(trunc, key=trunc.get)
over = {k: v for k, v in trunc.items() if v > TRUNCATION_BUDGET[k[1]]}
TRUNCATION = {"by_cell": {f"{m}|{c}": v for (m, c), v in trunc.items()},
              "budget": TRUNCATION_BUDGET,
              "worst": {"model_id": worst[0], "condition": worst[1], "rate": trunc[worst]},
              "biases_headline": "conservative"}
print(f"2. how hard the cap bound .................. worst {worst[0].split('/')[-1]}/"
      f"{worst[1]} at {trunc[worst]:.1%}")
for (m, cnd), v in sorted(trunc.items(), key=lambda kv: -kv[1])[:3]:
    print(f"     {m.split('/')[-1]:<24}{cnd:<16}{v:>6.1%}  (budget {TRUNCATION_BUDGET[cnd]:.0%})")
print("     truncation costs the condition it lands on, and it lands on the reasoning")
print("     conditions, so what is left understates bare and shrinks the headline.")
assert not over, (
    "truncated over budget: " + ", ".join(f"{c} on {m} at {v:.1%}" for (m, c), v in over.items())
    + f". Raise GEN_CAP (now {GEN_CAP}) and re-run -- the cap is in RUN_ID, so raising it "
    f"starts fresh rather than replaying these.")

# 3. the manipulation worked: the blurt produced no chain, the others did. Chain share
#    counts FORMATTING, not reasoning -- it is a blurt detector and nothing else.
print("\n3. did the manipulation take? (chain share, and the boxed rate it implies)")
print(f"   {'model':<22}{'condition':<16}{'chain':>8}{'boxed':>8}{'med tok':>9}")
for m in SWEPT:
    for c in CONDITIONS:
        rs = results[m][c]
        print(f"   {m.split('/')[-1]:<22}{c:<16}"
              f"{mean([len(segment_steps_cached(r)) >= 2 for r in rs]):>8.0%}"
              f"{mean([r['boxed'] for r in rs]):>8.0%}"
              f"{median([r['tokens_generated'] for r in rs]):>9.0f}")
print("   A low boxed rate under the blurt is expected: it was told not to write anything")
print("   else, so the grader falls back to the last number. That is the one place on this")
print("   page where the fallback does the work.")

# Part 2 — What the page shows

Every figure, number and gate the page renders, derived from what part 1 produced and written to
`out/page_05/`. No generation happens here, so these cells are cheap to re-run while a figure is being
adjusted.

## Figure 1 — two tabs on one problem

Two real completions, one problem, one model, one decoding setting.

| Tab | Condition | What it shows |
|---|---|---|
| left | `forced_blurt` | forbidden to reason: a number, and it is wrong |
| right | `bare` | left alone: it reasons, and it lands |

Curated from the sweep, never generated for the figure: the blurt must miss and bare must land.

The stepper walks the right tab a line at a time, and the chain it walks is the model's
**default** — nothing in the prompt asked for it.

In [ ]:
MAX_SPECIMEN_TOKENS = 260      # a chain the reader can walk on screen

blurt_by = {r["problem_id"]: r for r in results[PRIMARY]["forced_blurt"]}
cot_by   = {r["problem_id"]: r for r in results[PRIMARY]["zero_shot_cot"]}

candidates = [(blurt_by[b["problem_id"]], b, cot_by.get(b["problem_id"]))
              for b in results[PRIMARY]["bare"]
              if b["correct"] and not b["truncated"]
              and b["problem_id"] in blurt_by and not blurt_by[b["problem_id"]]["correct"]]


def _rank(t):
    _, bare, _cot = t
    return (bare["tokens_generated"] > MAX_SPECIMEN_TOKENS,
            bare["tokens_generated"],
            bare["problem_id"])


candidates.sort(key=_rank)
SPECIMEN_IS_FALLBACK = not candidates

if candidates:
    spec_blurt, spec_bare, spec_cot = candidates[0]
else:
    # Degrade and flag rather than fail: figures 2 and 3 are unaffected, and a flagged
    # specimen must not be translated (spec section 6).
    spec_bare = min((r for r in results[PRIMARY]["bare"] if not r["truncated"]),
                    key=lambda r: (not r["correct"], r["tokens_generated"], r["problem_id"]))
    spec_blurt = blurt_by[spec_bare["problem_id"]]
    spec_cot = cot_by.get(spec_bare["problem_id"])
    print("WARNING: no problem where the blurt misses and bare lands.\n"
          "         Falling back to an illustrative triple; derived.json flags it and the\n"
          "         run is not translatable. Raise n_items or run FULL.\n")

PROMPT_STEM = render_prompt(spec_bare["question"])
TABS = []
for name, rec in (("forced_blurt", spec_blurt), ("bare", spec_bare)):
    if rec is None:
        continue
    p = rec["prompt"]
    assert PROMPT_STEM in p, f"the {name} prompt does not contain the shared stem"
    cut = p.index(PROMPT_STEM)
    TABS.append({"condition": name, "label": LABEL[name],
                 "prompt_before": p[:cut],                        # empty on both here
                 "prompt_after": p[cut + len(PROMPT_STEM):],      # the suffix, highlighted
                 "trace": rec["trace"], "tokens": rec["tokens"],
                 "steps": segment_steps_cached(rec),
                 "tokens_generated": rec["tokens_generated"],
                 "extracted_answer": rec["extracted_answer"], "correct": rec["correct"]})

print(f"{len(candidates)} candidate problems"
      + ("   [FALLBACK SPECIMEN]" if SPECIMEN_IS_FALLBACK else "")
      + f";  chosen {spec_bare['problem_id']} (L{spec_bare['level']})\n")
print("QUESTION:", spec_bare["question"][:300])
print("TRUTH   :", spec_bare["ground_truth"], "\n")
for t in TABS:
    print(f"--- {t['label']:<26} {'correct' if t['correct'] else 'wrong  '}  "
          f"{t['tokens_generated']:>4} tokens, {len(t['steps'])} steps")
    print(f"    prompt adds: {t['prompt_after'].strip()[:88]!r}"
          if t["prompt_after"].strip() else "    prompt adds: nothing")
    for s in t["steps"][:3]:
        print("     ", s["text"][:86])
    if len(t["steps"]) > 3:
        print(f"      ... {len(t['steps']) - 3} more steps")
    print()

## Figure 2 — the cost of not reasoning

Two bars on one model, 95% Wilson intervals, the contrast tested **paired** with an exact McNemar
over the items where the conditions disagree. Every condition ran on the same problems, so an
unpaired test would throw that away.

The instruction condition is generated and reported but not plotted. Whether *asking* for
reasoning helps is a question about how the model was trained, and the deck answers that at
page 12; on this page it would be a third bar the reader has no framework to read.

In [ ]:
def by_item(model_id, cond):
    """problem_id -> was it right. Booleans, because that is what harness.mcnemar
    pairs on; anything needing the whole record reads `results` directly."""
    return {r["problem_id"]: bool(r["correct"]) for r in results[model_id][cond]}

def paired_ci(a, b, z=1.96):
    """95% interval on the paired difference in proportions (a minus b). This is what
    lets the page bound a null instead of asserting one."""
    both = set(a) & set(b)
    n = len(both)
    x = sum(1 for p in both if a[p] and not b[p])
    y = sum(1 for p in both if b[p] and not a[p])
    d = (x - y) / n
    se = sqrt(max(x + y - (x - y) ** 2 / n, 0.0)) / n
    return d, d - z * se, d + z * se

def summarise(model_id, cond):
    rs = results[model_id][cond]
    k = sum(r["correct"] for r in rs)
    lo, hi = wilson(k, len(rs))
    toks = sorted(r["tokens_generated"] for r in rs)
    return {"condition": cond, "label": LABEL[cond], "n": len(rs), "n_correct": k,
            "accuracy": k / len(rs), "ci": [lo, hi],
            "median_tokens": toks[len(toks) // 2], "mean_tokens": mean(toks),
            "p90_tokens": toks[int(len(toks) * 0.9)],
            "chain_share": mean([len(segment_steps_cached(r)) >= 2 for r in rs]),
            "boxed_rate": mean([r["boxed"] for r in rs])}

# Figure 2 plots what the page claims: what reasoning is worth. The instruction
# condition is still generated, still in traces.jsonl and still reported in
# derived.json -- it is simply not a bar. Why this model already reasons unprompted
# is a fact about its training, and training is page 12's subject, not this page's.
FIGURE_CONDITIONS = ["forced_blurt", "bare"]
SUMMARY = [summarise(PRIMARY, c) for c in FIGURE_CONDITIONS]
MEASURED_ONLY = [summarise(PRIMARY, c) for c in CONDITIONS if c not in FIGURE_CONDITIONS]
print(f"{PRIMARY}   n={len(items)}   MATH-500 L{'+L'.join(map(str, EVAL_LEVELS))}   greedy\n")
print(f"{'condition':<26}{'solved':>10}{'95% CI':>16}{'med tok':>10}{'chain':>8}")
for s in SUMMARY:
    ci = f"[{s['ci'][0]:.0%}, {s['ci'][1]:.0%}]"
    print(f"{s['label']:<26}{s['n_correct']:>4}/{s['n']:<5}{ci:>16}"
          f"{s['median_tokens']:>10.0f}{s['chain_share']:>8.0%}")

CONTRASTS = []
for hi_c, lo_c, what in (("bare", "forced_blurt", "cost of suppressing reasoning"),
                         ("zero_shot_cot", "bare", "gain from asking for reasoning")):
    a, b = by_item(PRIMARY, hi_c), by_item(PRIMARY, lo_c)
    x, y, p = harness.mcnemar(a, b)
    d, lo, hi = paired_ci(a, b)
    CONTRASTS.append({"high": hi_c, "low": lo_c, "what": what,
                      "delta_pts": d * 100, "ci_pts": [lo * 100, hi * 100],
                      "fixed": x, "broken": y, "p": p, "significant": p < 0.05})
    print(f"\n{what}:  {d*100:+.0f} points   95% CI [{lo*100:+.1f}, {hi*100:+.1f}]")
    print(f"   {x} problems fixed, {y} broken, p = {p:.4f}   "
          f"{'SIGNIFICANT' if p < 0.05 else 'not resolved'}")

SUPPRESSION_RESOLVES = CONTRASTS[0]["significant"]
_bound = max(abs(v) for v in CONTRASTS[1]["ci_pts"])
INSTRUCTION_NULL_BOUNDED = (not CONTRASTS[1]["significant"]) and _bound <= 15
NULL_BOUND_PTS = _bound

if not SUPPRESSION_RESOLVES:
    print("\n!! The suppression contrast did not resolve -- the page's core claim. This run is")
    print("   not translatable. Check blurt compliance above before blaming the sample size.")
if CONTRASTS[1]["significant"]:
    print(f"\n   The instruction contrast DID resolve. The page may not call it a null; report "
          f"the measured {CONTRASTS[1]['delta_pts']:+.0f} points instead.")
else:
    print(f"\n   The instruction contrast did not resolve, and is bounded within "
          f"+/-{_bound:.0f} points.")
    print(f"   It is measured, not plotted: page 12 is where a model reasoning")
    print(f"   unprompted gets explained. derived.json carries it for that callback.")

In [ ]:
import matplotlib.pyplot as plt

INK, FLOOR = "#3b4a6b", "#b23a3a"
fig, ax = plt.subplots(figsize=(8.4, 3.4))
ys = list(range(len(SUMMARY)))[::-1]
vals = [s["accuracy"] * 100 for s in SUMMARY]
err = [[(s["accuracy"] - s["ci"][0]) * 100 for s in SUMMARY],
       [(s["ci"][1] - s["accuracy"]) * 100 for s in SUMMARY]]

ax.barh(ys, vals, height=0.55,
        color=[FLOOR if s["condition"] == "forced_blurt" else INK for s in SUMMARY])
ax.errorbar(vals, ys, xerr=err, fmt="none", ecolor="#41474f", elinewidth=1.1, capsize=3)
ax.set_yticks(ys)
ax.set_yticklabels([s["label"] for s in SUMMARY])
ax.set_xlabel("share of problems solved (%)")
ax.set_xlim(0, 100)
ax.spines[["top", "right"]].set_visible(False)
for y, s in zip(ys, SUMMARY):
    ax.text(s["ci"][1] * 100 + 1.6, y, f"{s['accuracy']*100:.0f}%",
            va="center", fontsize=9, color="#41474f")
ax.set_title(f"{model_meta[PRIMARY]['label']} · greedy · MATH-500 "
             f"L{'+L'.join(map(str, EVAL_LEVELS))}, n={len(items)}",
             fontsize=10, loc="left", color="#555")

c0, c1 = CONTRASTS
fig.text(0.01, -0.06,
         f"suppression:   {c0['delta_pts']:+.0f} points   ·   "
         f"{c0['fixed']} fixed, {c0['broken']} broken   ·   p = {c0['p']:.4f}",
         fontsize=9, color="#555")
fig.text(0.01, -0.15,
         f"the instruction: {c1['delta_pts']:+.0f} points   ·   "
         f"{c1['fixed']} fixed, {c1['broken']} broken   ·   "
         f"95% CI [{c1['ci_pts'][0]:+.1f}, {c1['ci_pts'][1]:+.1f}]",
         fontsize=9, color="#555")
fig.text(0.01, -0.24, "Paired, exact McNemar over discordant items. Intervals on the bars are "
                      "95% Wilson.", fontsize=8.5, color="#8a5a2b")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_2.png", dpi=160, bbox_inches="tight")
plt.show()

## Figure 3 — the floor is flat

Two bars per model: gagged, and left alone. The right-hand bars fan out with capability. **The
left-hand bars do not.**

The claim is the flat floor, not a scaling law. Three points are not a law, model size appears on
no axis, and emergence in Wei et al. is a *sign flip* that this data does not contain — the trend
here is smooth and monotone. What the figure supports is a comparison of named checkpoints, and it
is tested rather than left to the eye (spec section 1).

In [ ]:
FLOOR_ROWS, CROSS = [], None
if len(SWEPT) > 1:
    for m in SWEPT:
        a, b = by_item(m, "bare"), by_item(m, "forced_blurt")
        x, y, p = harness.mcnemar(a, b)
        FLOOR_ROWS.append({
            "model_id": m, "label": model_meta[m]["label"], "params": model_meta[m]["params"],
            "quant": model_meta[m]["quant"],
            "blurt_accuracy": mean([r["correct"] for r in results[m]["forced_blurt"]]),
            "bare_accuracy": mean([r["correct"] for r in results[m]["bare"]]),
            "delta_pts": (mean([r["correct"] for r in results[m]["bare"]])
                          - mean([r["correct"] for r in results[m]["forced_blurt"]])) * 100,
            "fixed": x, "broken": y, "p": p})
    FLOOR_ROWS.sort(key=lambda r: r["params"])

    spread = (max(r["blurt_accuracy"] for r in FLOOR_ROWS)
              - min(r["blurt_accuracy"] for r in FLOOR_ROWS)) * 100
    FLOOR_IS_FLAT = spread <= 10

    print(f"{'model':<22}{'gagged':>9}{'alone':>9}{'delta':>8}{'fixed/broken':>15}{'p':>10}{'quant':>8}")
    for r in FLOOR_ROWS:
        flips = f"{r['fixed']}/{r['broken']}"
        print(f"{r['label']:<22}{r['blurt_accuracy']:>9.0%}{r['bare_accuracy']:>9.0%}"
              f"{r['delta_pts']:>+8.0f}{flips:>15}{r['p']:>10.4f}{r['quant']:>8}")
    print(f"\nthe floor spans {spread:.0f} points across "
          f"{FLOOR_ROWS[-1]['params']/FLOOR_ROWS[0]['params']:.1f}x the parameters   "
          f"-> {'FLAT' if FLOOR_IS_FLAT else 'not flat; drop the headline sentence'}")

    # The sentence the figure exists for, tested rather than eyeballed: the largest model
    # gagged against the smallest model left alone.
    big, small = FLOOR_ROWS[-1], FLOOR_ROWS[0]
    x, y, p = harness.mcnemar(by_item(small["model_id"], "bare"), by_item(big["model_id"], "forced_blurt"))
    CROSS = {"gagged_large": big["label"], "gagged_large_accuracy": big["blurt_accuracy"],
             "ungagged_small": small["label"], "ungagged_small_accuracy": small["bare_accuracy"],
             "delta_pts": (small["bare_accuracy"] - big["blurt_accuracy"]) * 100,
             "fixed": x, "broken": y, "p": p, "significant": p < 0.05,
             "param_ratio": big["params"] / small["params"]}
    print(f"\n{big['label']} gagged: {big['blurt_accuracy']:.0%}    "
          f"{small['label']} left alone: {small['bare_accuracy']:.0%}"
          f"    ({CROSS['delta_pts']:+.0f} points, p = {p:.4f})")
    if CROSS["significant"] and CROSS["delta_pts"] > 0:
        print(f"-> the chain is worth more than {CROSS['param_ratio']:.1f}x the parameters")
    else:
        print("-> that comparison did not resolve; the page must not make the sentence")
else:
    FLOOR_IS_FLAT = False
    print("one model swept; figure 3 is not drawn")

In [ ]:
if FLOOR_ROWS:
    fig, ax = plt.subplots(figsize=(7.6, 0.9 * len(FLOOR_ROWS) + 1.8))
    ys = list(range(len(FLOOR_ROWS)))[::-1]
    h = 0.34
    ax.barh([y + h / 2 for y in ys], [r["blurt_accuracy"] * 100 for r in FLOOR_ROWS],
            height=h, color=FLOOR, label="gagged")
    ax.barh([y - h / 2 for y in ys], [r["bare_accuracy"] * 100 for r in FLOOR_ROWS],
            height=h, color=INK, label="left alone")
    ax.set_yticks(ys)
    ax.set_yticklabels([f"{r['label']}\n{r['params']/1e9:.1f}B · {r['quant']}"
                        for r in FLOOR_ROWS], fontsize=8)
    ax.set_xlabel("share of problems solved (%)")
    ax.set_xlim(0, 100)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc="lower right")
    for y, r in zip(ys, FLOOR_ROWS):
        ax.text(r["blurt_accuracy"] * 100 + 1.4, y + h / 2, f"{r['blurt_accuracy']:.0%}",
                va="center", fontsize=8, color="#41474f")
        ax.text(r["bare_accuracy"] * 100 + 1.4, y - h / 2, f"{r['bare_accuracy']:.0%}",
                va="center", fontsize=8, color="#41474f")
    ax.set_title(f"gagged, every model lands in the same place · n={len(items)}",
                 fontsize=10, loc="left", color="#555")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "figure_3.png", dpi=160, bbox_inches="tight")
    plt.show()

## Independent verification

Everything above was computed from objects held in memory. This cell re-reads `traces.jsonl` from
disk and recomputes the two headline numbers from the raw generations alone, using nothing built
earlier. If the two disagree, the figures are wrong and the notebook says so rather than shipping
them.

This is what makes the page checkable rather than merely reproducible: a reader who distrusts the
derivation can run this cell against a trace file we published and get the same answer.

In [ ]:
raw = [json.loads(l) for l in TRACES_PATH.open(encoding="utf-8") if l.strip()]
raw = [r for r in raw if r["run_id"] == RUN_ID and r["model_id"] == PRIMARY]

acc = {}
for c in CONDITIONS:
    rs = [r for r in raw if r["condition"] == c]
    acc[c] = sum(r["correct"] for r in rs) / len(rs)

pairs = {}
for c in CONDITIONS:
    for r in raw:
        if r["condition"] == c:
            pairs.setdefault(r["problem_id"], {})[c] = r["correct"]
both = [v for v in pairs.values() if "bare" in v and "forced_blurt" in v]
fixed = sum(1 for v in both if v["bare"] and not v["forced_blurt"])
broken = sum(1 for v in both if v["forced_blurt"] and not v["bare"])

print("recomputed from traces.jsonl, independent of everything above:")
for c in CONDITIONS:
    print(f"  {LABEL[c]:<26} {acc[c]:>6.1%}")
print(f"  suppression                {(acc['bare'] - acc['forced_blurt']) * 100:>+6.1f} points "
      f"({fixed} fixed, {broken} broken, n={len(both)})")

assert abs(acc["bare"] - SUMMARY[1]["accuracy"]) < 1e-9, "figure 2 disagrees with the trace file"
assert fixed == CONTRASTS[0]["fixed"] and broken == CONTRASTS[0]["broken"], \
    "the paired counts disagree with the trace file"
print("\nOK   the figures above match the raw generations exactly")

## The close

What the chain cost, beside what it bought. This is the sentence page 6 opens on, and the numbers
are emitted so the handoff cannot borrow one the run did not earn.

In [ ]:
blurt, bare = SUMMARY[0], SUMMARY[1]
CLOSE = {
    "blurt_median_tokens": blurt["median_tokens"],
    "bare_median_tokens": bare["median_tokens"],
    "token_multiple": bare["median_tokens"] / max(blurt["median_tokens"], 1),
    "blurt_accuracy": blurt["accuracy"], "bare_accuracy": bare["accuracy"],
    "suppression_pts": CONTRASTS[0]["delta_pts"],
    "instruction_pts": CONTRASTS[1]["delta_pts"],
    "instruction_bound_pts": NULL_BOUND_PTS,
}
print(f"gagged      {CLOSE['blurt_median_tokens']:>5.0f} tokens   "
      f"{CLOSE['blurt_accuracy']:>5.0%} solved")
print(f"left alone  {CLOSE['bare_median_tokens']:>5.0f} tokens   "
      f"{CLOSE['bare_accuracy']:>5.0%} solved")
print(f"\n{CLOSE['token_multiple']:.0f}x the tokens for {CLOSE['suppression_pts']:+.0f} points."
      f"   <- the sentence page 6 opens on")

## Artifacts

`derived.json` is the contract with the page: the page reads it and nothing else, and **no
inference runs at demonstration time.** Its top-level `claims` object is the machine-readable form
of [spec section 6](../docs/page_5_refactor.md) and is read *before* any number.

`traces.jsonl` stays out of the web bundle and ships as a download beside the notebook, so the
verification cell above can be run by anyone.

In [ ]:
manifest = {
    "run_id": RUN_ID,
    "notebook": "page_05_how_to_reason.ipynb",
    "notebook_sha256": harness.file_sha256("page_05_how_to_reason.ipynb"),
    "harness": harness.fingerprint(),
    "page": 5, "run_mode": RUN_MODE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook_git_sha": harness.git_sha(),
    "primary_model": PRIMARY,
    "models": [model_meta[m] for m in SWEPT],
    "decoding": {"strategy": "greedy", "do_sample": False, "max_new_tokens": GEN_CAP,
                 "stop": ["\nQuestion:"]},
    "items": {"source": "MATH-500 (ch03 copy)", "eval_levels": list(EVAL_LEVELS),
              "n": len(items), "item_set_sha256": ITEM_SET_HASH, "seed": SEED},
    "conditions": {c: build_prompt(c, "<QUESTION>")[len(STEM):] for c in CONDITIONS},
    "grading": {"extract": "reasoning_from_scratch.ch03.extract_final_candidate",
                "grade": "reasoning_from_scratch.ch03.grade_answer"},
    "versions": {"torch": torch.__version__,
                 "transformers": __import__("transformers").__version__,
                 "device": str(device),
                 "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
    "integrity": {"blurt_compliance": BLURT_COMPLIANCE, "complies_under_tokens": COMPLIES_UNDER,
                  "truncation_by_cell": {f"{m}|{c}": v for (m, c), v in trunc.items()},
                  "truncation_budget": TRUNCATION_BUDGET,
                  "n_records": len(all_records)},
}

derived = {
    "manifest": manifest,
    # Read before any number: the machine-readable form of spec section 6.
    "truncation": TRUNCATION,
    "claims": {
        "suppression_resolves": SUPPRESSION_RESOLVES,
        "blurt_compliance": BLURT_COMPLIANCE >= 0.8,
        "specimen_usable": not SPECIMEN_IS_FALLBACK,
        "instruction_null_bounded": INSTRUCTION_NULL_BOUNDED,
        "floor_is_flat": bool(FLOOR_ROWS) and FLOOR_IS_FLAT,
        "may_compare_across_models": len(SWEPT) > 1,
    },
    "figure_1": {
        "problem_id": spec_bare["problem_id"], "level": spec_bare["level"],
        "question": spec_bare["question"], "ground_truth": spec_bare["ground_truth"],
        "prompt_stem": PROMPT_STEM, "tabs": TABS,
        "n_candidates": len(candidates), "specimen_is_fallback": SPECIMEN_IS_FALLBACK,
    },
    "figure_2": {
        "model_id": PRIMARY, "conditions": SUMMARY, "contrasts": CONTRASTS,
        "instruction_bound_pts": NULL_BOUND_PTS,
        # Generated and graded, deliberately not plotted: a model that reasons unprompted
        # does so because of its training, which is page 12's subject. Page 12 opens on
        # this number, so it is published rather than dropped. See spec section 5.
        "measured_only": MEASURED_ONLY,
    },
    "figure_3": {"models": FLOOR_ROWS, "cross": CROSS,
                 "floor_is_flat": bool(FLOOR_ROWS) and FLOOR_IS_FLAT} if FLOOR_ROWS else None,
    "close": CLOSE,
}

DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"derived.json   {harness.kb(DERIVED_PATH):>9.1f} KB   <- bundled with the page")
print(f"manifest.json  {harness.kb(MANIFEST_PATH):>9.1f} KB   <- bundled with the page")
print(f"traces.jsonl   {harness.kb(TRACES_PATH):>9.1f} KB   <- published as a download")

blocking = ["suppression_resolves", "blurt_compliance", "specimen_usable"]
failed = [k for k in blocking if not derived["claims"][k]]
print(f"\n{'TRANSLATABLE' if not failed else 'NOT TRANSLATABLE: ' + ', '.join(failed)}")
for k, v in derived["claims"].items():
    print(f"   {'ok ' if v else 'NO '} {k}")

## Notes for re-derivation

- Greedy decoding diverges on a handful of items across hardware (`get_device` enables TF32 on
  CUDA): expect the shape and the ranking to reproduce, not every digit.
- Suppression is a large effect and survives small runs; `QUICK` at *n* = 50 resolves it. The
  instruction contrast is a *bound*, and the bound is looser at smaller *n* — at 50 items it is
  roughly ±14 points, at 100 roughly ±10.
- Chain share counts formatting, not reasoning. It is used here as a blurt detector and to place
  the reveal boundaries, and it must not be reported as a measure of reasoning (spec section 2).
- `forced_blurt` is one phrasing of "do not explain". A different phrasing would move the floor,
  and the exact string used is in `manifest.conditions`.
- The blurt condition emits almost no `\boxed{}` by construction, so the grader's fallback to the
  last number does the extraction there. Reported per condition in the integrity section.
- Changing model, band, cap or item count changes `RUN_ID`; old traces stay in the file as a
  record of what was run and are never replayed.
- These numbers should agree with `experiment_3`'s on the overlapping cells, since the conditions
  and the band match. Disagreement beyond a couple of items means one of the two is wrong.

What the page may claim is [`docs/page_5_refactor.md`](../docs/page_5_refactor.md) section 6,
checked at translation ([`docs/page_6_refactor.md`](../docs/page_6_refactor.md) section 6).